# Train LoRA checkpoints on Colab
Select a GPU runtime and run all cells. The defaults train nine FP16 adapters for the local full sweep, one for each combination of three seeds and three training sizes. Training runs on the VM. Each completed checkpoint is published to Google Drive. The notebook uses MBPP and the base model; it does no benchmark generation or scoring.

For the Mac CLI run, the pinned checkout’s `configs/full.toml` supplies the exact target configuration. To target a run with different settings (including a run from **Full_Sweep.ipynb**), set `TARGET_MANIFEST` to its saved `manifest.json` and use its code revision. The adapter identity includes the **entire** config, selected examples, validation rows, and seed. Filtering jobs below leaves that identity intact. Matching seeds reproduce example selection and the dialect; floating-point training can differ across GPU backends.


In [ ]:
REVISION = "38daf7cd68699438c0a5b6f6400877572f4c0137"
TARGET_MANIFEST = None  # Example: "/content/drive/MyDrive/cs182-sweep/runs/<name>/manifest.json"
SEEDS = None  # None selects all configured seeds; e.g. [0] for one shard
SIZES = None  # None selects all configured sizes; e.g. [16, 64]

import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import tomllib
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/cs182-sweep")
WORK = Path("/content/cs182-lora-training")
WORK.mkdir(parents=True, exist_ok=True)
repo_url = "https://github.com/KG2468/ICLvsLoRA.git"
repo = WORK / "repo"
if not repo.exists():
    source = DRIVE / "repo"
    subprocess.run(["git", "clone", "--no-checkout", str(source) if (source / ".git").exists() else repo_url, str(repo)], check=True)
if subprocess.run(["git", "-C", str(repo), "cat-file", "-e", REVISION + "^{commit}"], capture_output=True).returncode:
    subprocess.run(["git", "-C", str(repo), "fetch", repo_url, REVISION], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", REVISION], check=True)
manifest = json.loads(Path(TARGET_MANIFEST).read_text()) if TARGET_MANIFEST else None
cfg = manifest["config"] if manifest else tomllib.loads((repo / "configs/full.toml").read_text())
exp = cfg["experiment"]
def select(values, requested):
    if requested is None:
        return values
    if not requested or not set(requested).issubset(values):
        raise ValueError(f"Select a nonempty subset of {values}")
    return [value for value in values if value in requested]

jobs = [(seed, n) for seed in select(exp["seeds"], SEEDS) for n in select(exp["sizes"], SIZES)]
model_dir = Path(cfg["model"]["directory"]) / cfg["model"]["precision"]
data_dir = Path(exp["data_dir"])
run_dir = Path(exp["output_dir"]) / exp["name"]
for path in (data_dir, model_dir, run_dir):
    if path.is_absolute() or ".." in path.parts:
        raise ValueError(f"Target config requires project-relative paths: {path}")
destination = DRIVE / run_dir / "adapters"
print("Jobs (seed, N):", jobs)
print("Drive checkpoint directory:", destination)


## Inventory checkpoints before starting the GPU
The notebook uses MBPP to reconstruct the trainer’s job identities, then checks the saved weights on Drive. A verified job is skipped. Fully written checkpoints left in staging or on a surviving VM can also be recovered.


In [ ]:
if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)
uv = shutil.which("uv") or str(Path(sys.executable).parent / "uv")
venv = WORK / "venv"
subprocess.run([uv, "python", "install", "3.12.11"], check=True)
py = venv / "bin/python"
if not py.exists():
    subprocess.run([uv, "venv", "--python", "3.12.11", str(venv)], check=True)
env = WORK / "venv"
subprocess.run([uv, "python", "install", "3.12.11"], check=True)
py = venv / "bin/python"
if not py.exists():
    subprocess.run([uv, "venv", "--python", "3.12.11", str(venv)], check=True)
subprocess.run([uv, "pip", "install", "--python", str(py), mlx, "mlx-lm[train]==0.32.2",
                "datasets==4.8.5", "transformers==5.23.0", "huggingface-hub==1.33.0"], check=True)
env = {**os.environ, "PYTHONPATH": str(repo / "src"), "PYTHONUNBUFFERED": "1"}
def run(*args, cwd=WORK):
    command = [str(py), *args]
    with subprocess.Popen(command, env=env, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, errors="replace") as child:
        try:
            for line in child.stdout:
                print(line, end="", flush=True)
        except BaseException:
            child.kill()
            raise
        if child.wait():
            raise subprocess.CalledProcessError(child.returncode, command)
subprocess.run([uv, "pip", "install", "--python", str(py), "datasets==4.8.5", "huggingface-hub==1.33.0"], check=True)
settings = WORK / "target-config.json"
settings.write_text(json.dumps(cfg))
local_data = WORK / data_dir
local_data.mkdir(parents=True, exist_ok=True)
for name in ("mbpp_train.jsonl", "mbpp_valid.jsonl", "manifest.json"):
    cached = DRIVE / data_dir / name
    if cached.is_file() and not (local_data / name).exists():
        shutil.copy2(cached, local_data / name)
load_cfg = "import json, sys; cfg = json.load(open(sys.argv[1])); "
run("-c", load_cfg + "from final_project.data import prepare; cfg['experiment']['benchmarks'] = []; prepare(cfg)", str(settings))
def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

if manifest:
    for key in ("mbpp_train", "mbpp_valid"):
        assert sha256(local_data / (key + ".jsonl")) == manifest["datasets"][key], f"Dataset mismatch: {key}"

identity_script = """import json, sys
from final_project.config import digest
from final_project.data import examples
from final_project.language import Language
cfg = json.load(open(sys.argv[1]))
jobs = json.loads(sys.argv[2])
identities = {}
for seed in sorted({s for s, _ in jobs}):
    rows, valid = examples(cfg, Language(seed), seed)
    for s, n in jobs:
        if s == seed:
            identities[f"seed{seed}-n{n}"] = digest({"cfg": cfg, "rows": rows[:n], "valid": valid, "seed": seed})
print(json.dumps(identities))"""
identities = json.loads(subprocess.check_output(
    [str(py), "-c", identity_script, str(settings), json.dumps(jobs)], cwd=WORK, env=env, text=True))
files = ("adapter_config.json", "adapters.safetensors", "complete.json")

def inspect_checkpoint(directory, label):
    if not directory.exists():
        return "missing"
    try:
        marker = json.loads((directory / "complete.json").read_text())
        if not isinstance(marker, dict) or not marker.get("identity"):
            return "incomplete"
        if marker["identity"] != identities[label]:
            return "different job"
        config = json.loads((directory / "adapter_config.json").read_text())
        if not isinstance(config, dict) or not config:
            return "incomplete"
        if sha256(directory / "adapters.safetensors") != marker.get("weights_sha256"):
            return "checksum mismatch"
    except (FileNotFoundError, ValueError):
        return "incomplete"
    return "verified"

def publish(source, label):
    from uuid import uuid4
    saved = destination / label
    with tempfile.TemporaryDirectory(dir=destination, prefix=".staging-") as scratch:
        staged = Path(scratch) / label
        staged.mkdir()
        for filename in files:
            shutil.copy2(source / filename, staged / filename)
        if inspect_checkpoint(staged, label) != "verified":
            raise ValueError(f"Copied checkpoint failed verification: {label}")
        prior = inspect_checkpoint(saved, label)
        if prior == "verified":
            return
        if prior == "different job":
            raise ValueError(f"Checkpoint identity conflict: {saved}")
        if saved.exists():
            archive = destination / f".incomplete-{label}-{uuid4().hex}"
            saved.rename(archive)
            print("Preserved previous incomplete checkpoint:", archive)
        staged.rename(saved)

def pending_jobs():
    destination.mkdir(parents=True, exist_ok=True)
    pending = []
    for seed, n in jobs:
        label = f"seed{seed}-n{n}"
        saved = destination / label
        status = inspect_checkpoint(saved, label)
        if status == "different job":
            raise ValueError(f"Different configuration or examples in {saved}. Check TARGET_MANIFEST.")
        if status != "verified":
            candidates = [*destination.glob(f".staging-*/{label}"), *WORK.glob(f"tmp*/{label}")]
            for candidate in candidates:
                if inspect_checkpoint(candidate, label) == "verified":
                    publish(candidate, label)
                    status = "recovered"
                    break
            else:
                pending.append((seed, n))
        print(f"{label:16} {status}")
    print(f"{len(jobs) - len(pending)}/{len(jobs)} complete; {len(pending)} pending.", flush=True)
    return pending

pending = pending_jobs()


## Set up the GPU and stage the model for pending jobs
If every selected checkpoint is verified, this cell skips GPU setup and model preparation.


In [ ]:
if pending:
    if not shutil.which("nvidia-smi"):
        raise RuntimeError("Select a Colab NVIDIA GPU runtime and rerun.")
    gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,driver_version,compute_cap",
                                   "--format=csv,noheader"], text=True).splitlines()[0]
    name, version, capability = [part.strip() for part in gpu.split(",")]
    if float(capability) < 7.5 or tuple(map(int, version.split("."))) < (550, 54, 14):
        raise RuntimeError(f"Unsupported GPU or driver: {gpu}")
    mlx = "mlx[cuda13]==0.32.4" if int(version.split(".")[0]) >= 580 else "mlx[cuda12]==0.32.4"
    run("-c", "import mlx.core as mx; mx.set_default_device(mx.gpu); print(mx.device_info(), mx.ones(1).item())")

    local_model = WORK / model_dir
    if not local_model.exists():
        local_model.parent.mkdir(parents=True, exist_ok=True)
        with tempfile.TemporaryDirectory(dir=WORK) as tmp:
            staged = Path(tmp) / model_dir
            cached = DRIVE / model_dir
            if cached.is_dir():
                shutil.copytree(cached, staged)
            run("-c", load_cfg + "from final_project.backend import convert; convert(cfg)", str(settings), cwd=tmp)
            staged.rename(local_model)
    else:
        run("-c", load_cfg + "from final_project.backend import verify_model; verify_model(cfg)", str(settings))

    if manifest:
        for name, expected in manifest["weights"].items():
            assert sha256(local_model / name) == expected, f"Model mismatch: {name}"
else:
    print("All selected checkpoints are ready.")


## Train pending jobs and publish each checkpoint
Rerunning this cell checks Drive again before launching a trainer. Each run publishes on completion. A job interrupted before its final checkpoint starts again from its first update. Use disjoint jobs across training VMs.


In [ ]:
for index, (seed, n) in enumerate(pending_jobs(), 1):
    label = f"seed{seed}-n{n}"
    print(f"[{index}] Starting {label}", flush=True)
    with tempfile.TemporaryDirectory(dir=WORK) as tmp:
        adapter = Path(tmp) / label
        job = Path(tmp) / "job.json"
        job.write_text(json.dumps({"cfg": cfg, "seed": seed, "n": n, "adapter": str(adapter)}))
        run("-m", "final_project", "_train", str(job))
        print(f"{label} — saving and verifying on Drive", flush=True)
        publish(adapter, label)
        print(f"{label} — complete", flush=True)
print(f"{len(jobs)} requested checkpoints verified.")


## Use the adapters in the evaluation run
For a Mac run, copy the `seed…-n…` folders from the printed Drive directory into `<project>/runs/<target experiment name>/adapters/` before its LoRA stage starts. For a fresh Colab full sweep using the same target manifest, the folders are already in the Drive restore layout. If that evaluator is already running, copy newly completed folders into its local run directory before its LoRA stage. The evaluator validates the saved identity and weight checksum when it reuses each adapter.
